In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
 
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
 
OUTPUT_DIR = "Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
print("\n" + "="*70)
print("STEP 1 — LOADING CLEANED DATA")
print("="*70)
 
loyalty  = pd.read_csv(f"{OUTPUT_DIR}/loyalty_cleaned_v2.csv")
flights  = pd.read_csv(f"{OUTPUT_DIR}/flights_cleaned_v2.csv")
 
# Re-parse activity_date
flights["activity_date"] = pd.to_datetime(flights["activity_date"])
 
print(f"  Loyalty : {loyalty.shape}")
print(f"  Flights : {flights.shape}")
print(f"  Flight date range: {flights['activity_date'].min().date()} → "
      f"{flights['activity_date'].max().date()}")
 
# Identify key column names
ID_COL      = "loyalty_number"
FLIGHTS_COL = "total_flights"
DIST_COL    = "distance"
PTS_EARN    = "points_accumulated"
PTS_REDEEM  = "points_redeemed"
DOLLAR_PTS  = "dollar_cost_points_redeemed"
YEAR_COL    = "year"
MONTH_COL   = "month"
CLV_COL     = "clv"
 
 


STEP 1 — LOADING CLEANED DATA
  Loyalty : (16737, 22)
  Flights : (389065, 10)
  Flight date range: 2017-01-01 → 2018-12-01


In [3]:
print("\n" + "="*70)
print("STEP 2 — DEFINING CHURN LABELS")
print("="*70)
 
OBSERVATION_END = pd.Timestamp("2018-06-01")   # last month of feature window
CHURN_START     = pd.Timestamp("2018-07-01")   # start of churn window
CHURN_END       = pd.Timestamp("2018-12-01")   # end of churn window
 
# Flights in the churn window (Jul–Dec 2018)
churn_window = flights[flights["activity_date"] >= CHURN_START]
flights_in_churn = (churn_window
                    .groupby(ID_COL)[FLIGHTS_COL]
                    .sum()
                    .reset_index()
                    .rename(columns={FLIGHTS_COL: "flights_churn_window"}))
 
# Flights in observation window (Jan 2017 – Jun 2018)
obs_window = flights[flights["activity_date"] <= OBSERVATION_END]
flights_in_obs = (obs_window
                  .groupby(ID_COL)[FLIGHTS_COL]
                  .sum()
                  .reset_index()
                  .rename(columns={FLIGHTS_COL: "flights_obs_window"}))
 
# Merge both into loyalty
labels = loyalty[[ID_COL]].copy()
labels = labels.merge(flights_in_obs,   on=ID_COL, how="left")
labels = labels.merge(flights_in_churn, on=ID_COL, how="left")
labels["flights_obs_window"]   = labels["flights_obs_window"].fillna(0)
labels["flights_churn_window"] = labels["flights_churn_window"].fillna(0)
 
# Assign churn label
def assign_churn(row):
    if row["flights_obs_window"] == 0 and row["flights_churn_window"] == 0:
        return 2   # Never flew — dormant (separate segment)
    elif row["flights_churn_window"] == 0:
        return 1   # Was active before, went silent → CHURNED
    else:
        return 0   # Flew in churn window → ACTIVE
 
labels["churn_label"] = labels.apply(assign_churn, axis=1)
 
# Summary
label_counts = labels["churn_label"].value_counts().sort_index()
label_map    = {0: "Active", 1: "Churned", 2: "Dormant (never flew)"}
print("\n  Churn Label Distribution:")
for k, v in label_counts.items():
    print(f"    {label_map[k]:25s}: {v:,}  ({v/len(labels)*100:.1f}%)")
 
# Save label summary
label_summary = pd.DataFrame({
    "label_code":  label_counts.index,
    "label_name":  [label_map[k] for k in label_counts.index],
    "count":       label_counts.values,
    "pct":         (label_counts.values / len(labels) * 100).round(2)
})
label_summary.to_csv(f"{OUTPUT_DIR}/churn_label_summary.csv", index=False)
 
print(f"""
  CHURN DEFINITION (documented for report):
  ──────────────────────────────────────────
  Observation window : Jan 2017 – Jun 2018 (features built from this)
  Churn window       : Jul 2018 – Dec 2018 (label derived from this)
  Churned  (1)       : Active in obs window, zero flights in churn window
  Active   (0)       : At least 1 flight in churn window
  Dormant  (2)       : Zero flights across entire 2017–2018 period
  
  Rationale: 6-month silence is a strong disengagement signal for airlines.
  Seasonal gaps of 1-3 months are normal; 6 consecutive months is not.
  We separate 'dormant' from 'churned' to avoid penalising never-active members.
""")


STEP 2 — DEFINING CHURN LABELS

  Churn Label Distribution:
    Active                   : 14,268  (85.2%)
    Churned                  : 899  (5.4%)
    Dormant (never flew)     : 1,570  (9.4%)

  CHURN DEFINITION (documented for report):
  ──────────────────────────────────────────
  Observation window : Jan 2017 – Jun 2018 (features built from this)
  Churn window       : Jul 2018 – Dec 2018 (label derived from this)
  Churned  (1)       : Active in obs window, zero flights in churn window
  Active   (0)       : At least 1 flight in churn window
  Dormant  (2)       : Zero flights across entire 2017–2018 period
  
  Rationale: 6-month silence is a strong disengagement signal for airlines.
  Seasonal gaps of 1-3 months are normal; 6 consecutive months is not.
  We separate 'dormant' from 'churned' to avoid penalising never-active members.



In [4]:
print("="*70)
print("STEP 3 — FEATURE ENGINEERING (observation window only)")
print("="*70)
 
obs = obs_window.copy()   # Jan 2017 – Jun 2018 only
 
# ── Helper: per-member aggregation ───────────────────────────────────────────
def member_agg(df, months, suffix):
    """Aggregate flight metrics for the last N months before OBSERVATION_END."""
    cutoff = OBSERVATION_END - pd.DateOffset(months=months) + pd.DateOffset(months=1)
    subset = df[df["activity_date"] >= cutoff]
    agg = subset.groupby(ID_COL).agg(
        flights      = (FLIGHTS_COL, "sum"),
        distance     = (DIST_COL,    "sum"),
        pts_earned   = (PTS_EARN,    "sum"),
        pts_redeemed = (PTS_REDEEM,  "sum"),
        active_months= (FLIGHTS_COL, lambda x: (x > 0).sum()),
    ).reset_index()
    agg.columns = [ID_COL] + [f"{c}_{suffix}" for c in agg.columns[1:]]
    return agg
 
# Aggregate over 3, 6, 12, and 18 month windows
agg_3m  = member_agg(obs, 3,  "3m")
agg_6m  = member_agg(obs, 6,  "6m")
agg_12m = member_agg(obs, 12, "12m")
agg_18m = member_agg(obs, 18, "18m")   # full observation window
 
print("  Built aggregations for 3m, 6m, 12m, 18m windows ✓")
 
# ── Recency: months since last flight ────────────────────────────────────────
last_flight = (obs[obs[FLIGHTS_COL] > 0]
               .groupby(ID_COL)["activity_date"]
               .max()
               .reset_index()
               .rename(columns={"activity_date": "last_flight_date"}))
 
last_flight["months_since_last_flight"] = (
    (OBSERVATION_END.year  - last_flight["last_flight_date"].dt.year) * 12
    + (OBSERVATION_END.month - last_flight["last_flight_date"].dt.month)
)
 
print("  Built recency feature (months_since_last_flight) ✓")
 
# ── Trend: flight trajectory (recent vs earlier) ─────────────────────────────
# Trend = flights_3m  minus flights in the 3 months before that (months 4–6)
cutoff_3m = OBSERVATION_END - pd.DateOffset(months=3) + pd.DateOffset(months=1)
cutoff_6m = OBSERVATION_END - pd.DateOffset(months=6) + pd.DateOffset(months=1)
 
period_recent = (obs[obs["activity_date"] >= cutoff_3m]
                 .groupby(ID_COL)[FLIGHTS_COL].sum()
                 .reset_index().rename(columns={FLIGHTS_COL: "flights_recent_3m"}))
 
period_prior  = (obs[(obs["activity_date"] >= cutoff_6m) &
                     (obs["activity_date"] <  cutoff_3m)]
                 .groupby(ID_COL)[FLIGHTS_COL].sum()
                 .reset_index().rename(columns={FLIGHTS_COL: "flights_prior_3m"}))
 
trend = period_recent.merge(period_prior, on=ID_COL, how="outer").fillna(0)
trend["flight_trend"] = trend["flights_recent_3m"] - trend["flights_prior_3m"]
# Negative trend = flying less recently; positive = flying more
trend = trend[[ID_COL, "flight_trend"]]
print("  Built trend feature (flight_trend = recent 3m minus prior 3m) ✓")
 
# ── Redemption ratio ─────────────────────────────────────────────────────────
rdm = agg_18m[[ID_COL, "pts_earned_18m", "pts_redeemed_18m"]].copy()
rdm["redemption_ratio"] = np.where(
    rdm["pts_earned_18m"] > 0,
    rdm["pts_redeemed_18m"] / rdm["pts_earned_18m"],
    0
)
rdm["redemption_ratio"] = rdm["redemption_ratio"].clip(0, 1)
rdm = rdm[[ID_COL, "redemption_ratio"]]
print("  Built redemption_ratio (points redeemed / points earned, capped 0–1) ✓")
 
# ── Seasonal consistency ─────────────────────────────────────────────────────
# How many distinct seasons did the member fly in?
# Map months to seasons
def month_to_season(m):
    if m in [12, 1, 2]:  return "Winter"
    if m in [3, 4, 5]:   return "Spring"
    if m in [6, 7, 8]:   return "Summer"
    return "Fall"
 
obs["season"] = obs[MONTH_COL].apply(month_to_season)
season_diversity = (obs[obs[FLIGHTS_COL] > 0]
                    .groupby(ID_COL)["season"]
                    .nunique()
                    .reset_index()
                    .rename(columns={"season": "seasons_flown"}))
# 1 = flies only in one season (risky); 4 = year-round flyer (more loyal)
print("  Built seasons_flown (1=single season, 4=year-round) ✓")
 
# ── Enrolment tenure ─────────────────────────────────────────────────────────
# Months since enrolment (as of Jun 2018)
loyalty["enrollment_tenure_months"] = (
    (2018 - loyalty["enrollment_year"]) * 12
    + (6   - loyalty["enrollment_month"])
).clip(lower=0)
print("  Built enrollment_tenure_months ✓")
 
# ── Card tier encoding ────────────────────────────────────────────────────────
TIER_COL = "loyalty_card"
tier_order = {"Star": 0, "Nova": 1, "Aurora": 2}   # adjust if yours differ
 
if TIER_COL in loyalty.columns:
    actual_tiers = loyalty[TIER_COL].unique()
    print(f"\n  Card tiers found: {sorted(actual_tiers)}")
    # Auto-build ordinal mapping by alphabetical order if names differ
    if not all(t in tier_order for t in actual_tiers):
        sorted_tiers = sorted([t for t in actual_tiers if pd.notna(t)])
        tier_order   = {t: i for i, t in enumerate(sorted_tiers)}
        print(f"  Auto-built tier encoding: {tier_order}")
    loyalty["card_tier_encoded"] = loyalty[TIER_COL].map(tier_order).fillna(0)
    print("  Built card_tier_encoded ✓")
 
# ── Salary encoding ──────────────────────────────────────────────────────────
# Salary is a continuous numeric column (float). 'Unknown' was filled in Phase 1
# for missing values. We convert to numeric and impute Unknown with median.
SAL_COL = "salary"
if SAL_COL in loyalty.columns:
    # Convert to numeric; 'Unknown' string becomes NaN
    salary_numeric = pd.to_numeric(loyalty[SAL_COL], errors="coerce")
    # Also flag negative salaries (anomaly found in phase1b)
    n_neg_sal = (salary_numeric < 0).sum()
    if n_neg_sal > 0:
        print(f"\n  WARNING: {n_neg_sal} negative salary values → set to NaN")
        salary_numeric[salary_numeric < 0] = np.nan
    sal_median = salary_numeric.median()
    loyalty["salary_encoded"] = salary_numeric.fillna(sal_median)
    n_unknown = salary_numeric.isna().sum()
    print(f"\n  Salary: {n_unknown} Unknown/NaN → imputed with median ({sal_median:,.0f})")
    print(f"  salary_encoded range: {loyalty['salary_encoded'].min():,.0f} – {loyalty['salary_encoded'].max():,.0f}")
    print("  Built salary_encoded ✓")
 
# ── Gender & Marital status (one-hot) ─────────────────────────────────────────
cat_cols = []
for col in ["gender", "marital_status"]:
    if col in loyalty.columns:
        dummies = pd.get_dummies(loyalty[col], prefix=col, drop_first=True, dtype=int)
        loyalty = pd.concat([loyalty, dummies], axis=1)
        cat_cols.extend(dummies.columns.tolist())
        print(f"  One-hot encoded '{col}' → {dummies.columns.tolist()} ✓")
 
# ── Education encoding ────────────────────────────────────────────────────────
# Ordinal encoding: higher number = higher education level
# High School or Below < College < Bachelor < Master < Doctor
EDU_COL = "education"
if EDU_COL in loyalty.columns:
    edu_vals = loyalty[EDU_COL].unique()
    print(f"\n  Education values found: {sorted(str(e) for e in edu_vals)}")
    edu_order = {
        "High School or Below": 1,
        "College":              2,
        "Bachelor":             3,
        "Master":               4,
        "Doctor":               5,
        "Unknown":              0   # unknown = separate from all levels
    }
    # Check all values are covered; warn if not
    missing_edu = [str(e) for e in edu_vals if str(e) not in edu_order]
    if missing_edu:
        print(f"  WARNING: Unmapped education values: {missing_edu} → set to 0")
        for v in missing_edu:
            edu_order[v] = 0
    print(f"  Education encoding used: {edu_order}")
    loyalty["education_encoded"] = loyalty[EDU_COL].astype(str).map(edu_order).fillna(0).astype(int)
    print("  Built education_encoded ✓")
 
 

STEP 3 — FEATURE ENGINEERING (observation window only)
  Built aggregations for 3m, 6m, 12m, 18m windows ✓
  Built recency feature (months_since_last_flight) ✓
  Built trend feature (flight_trend = recent 3m minus prior 3m) ✓
  Built redemption_ratio (points redeemed / points earned, capped 0–1) ✓
  Built seasons_flown (1=single season, 4=year-round) ✓
  Built enrollment_tenure_months ✓

  Card tiers found: ['Aurora', 'Nova', 'Star']
  Built card_tier_encoded ✓


  Salary: 4258 Unknown/NaN → imputed with median (73,510)
  salary_encoded range: 15,609 – 407,228
  Built salary_encoded ✓
  One-hot encoded 'gender' → ['gender_Male'] ✓
  One-hot encoded 'marital_status' → ['marital_status_Married', 'marital_status_Single'] ✓

  Education values found: ['Bachelor', 'College', 'Doctor', 'High School or Below', 'Master']
  Education encoding used: {'High School or Below': 1, 'College': 2, 'Bachelor': 3, 'Master': 4, 'Doctor': 5, 'Unknown': 0}
  Built education_encoded ✓


In [5]:
print("\n" + "="*70)
print("STEP 4 — ASSEMBLING MASTER FEATURE TABLE")
print("="*70)
 
# Start with loyalty demographics
demo_cols = [
    ID_COL, CLV_COL,
    "enrollment_tenure_months",
    "card_tier_encoded",
    "salary_encoded",
    "education_encoded",
    "never_flew_flag",
] + cat_cols
 
# Keep only columns that exist
demo_cols = [c for c in demo_cols if c in loyalty.columns]
features  = loyalty[demo_cols].copy()
 
# Merge flight aggregations
for agg_df in [agg_3m, agg_6m, agg_12m, agg_18m]:
    features = features.merge(agg_df, on=ID_COL, how="left")
 
# Merge derived features
for df in [last_flight[[ID_COL, "months_since_last_flight"]],
           trend,
           rdm,
           season_diversity]:
    features = features.merge(df, on=ID_COL, how="left")
 
# Merge churn labels
features = features.merge(
    labels[[ID_COL, "churn_label", "flights_obs_window", "flights_churn_window"]],
    on=ID_COL, how="left"
)
 
# Fill NaN for members with no flights (they get 0 for all activity features)
flight_feature_cols = [c for c in features.columns
                       if any(s in c for s in
                              ["flights_", "distance_", "pts_", "active_months_",
                               "months_since", "flight_trend", "redemption",
                               "seasons_flown"])]
features[flight_feature_cols] = features[flight_feature_cols].fillna(0)
 
# months_since_last_flight: if never flew, set to max (18 = full obs window)
features["months_since_last_flight"] = features["months_since_last_flight"].fillna(18)
 
print(f"\n  Feature table shape: {features.shape}")
print(f"\n  All columns:")
for col in features.columns:
    n_null = features[col].isnull().sum()
    null_str = f"  ← {n_null} nulls" if n_null > 0 else ""
    print(f"    {col}{null_str}")
 


STEP 4 — ASSEMBLING MASTER FEATURE TABLE

  Feature table shape: (16737, 37)

  All columns:
    loyalty_number
    clv
    enrollment_tenure_months
    card_tier_encoded
    salary_encoded
    education_encoded
    never_flew_flag
    gender_Male
    marital_status_Married
    marital_status_Single
    flights_3m
    distance_3m
    pts_earned_3m
    pts_redeemed_3m
    active_months_3m
    flights_6m
    distance_6m
    pts_earned_6m
    pts_redeemed_6m
    active_months_6m
    flights_12m
    distance_12m
    pts_earned_12m
    pts_redeemed_12m
    active_months_12m
    flights_18m
    distance_18m
    pts_earned_18m
    pts_redeemed_18m
    active_months_18m
    months_since_last_flight
    flight_trend
    redemption_ratio
    seasons_flown
    churn_label
    flights_obs_window
    flights_churn_window


In [6]:
print("\n" + "="*70)
print("STEP 5 — FEATURE SANITY CHECKS")
print("="*70)
 
# Check: churn rate per tier
if "card_tier_encoded" in features.columns:
    churn_by_tier = (features[features["churn_label"] < 2]
                     .groupby("card_tier_encoded")["churn_label"]
                     .agg(["mean", "count"])
                     .rename(columns={"mean": "churn_rate", "count": "n_members"}))
    print("\n  Churn rate by card tier (0=base, higher=premium):")
    print(churn_by_tier.round(3).to_string())
 
# Check: churn rate by recency bucket
features["recency_bucket"] = pd.cut(
    features["months_since_last_flight"],
    bins=[-1, 1, 3, 6, 12, 18],
    labels=["0-1m", "1-3m", "3-6m", "6-12m", "12-18m"]
)
churn_by_recency = (features[features["churn_label"] < 2]
                    .groupby("recency_bucket", observed=True)["churn_label"]
                    .agg(["mean", "count"])
                    .rename(columns={"mean": "churn_rate", "count": "n_members"}))
print("\n  Churn rate by months since last flight:")
print(churn_by_recency.round(3).to_string())
 
# Check: mean CLV by churn label
clv_by_churn = (features.groupby("churn_label")[CLV_COL]
                .agg(["mean", "median", "count"])
                .round(2))
label_name_map = {0: "Active", 1: "Churned", 2: "Dormant"}
clv_by_churn.index = clv_by_churn.index.map(label_name_map)
print("\n  CLV by churn label:")
print(clv_by_churn.to_string())
 


STEP 5 — FEATURE SANITY CHECKS

  Churn rate by card tier (0=base, higher=premium):
                   churn_rate  n_members
card_tier_encoded                       
0                       0.056       6934
1                       0.063       5131
2                       0.061       3102

  Churn rate by months since last flight:
                churn_rate  n_members
recency_bucket                       
0-1m                 0.011      12509
1-3m                 0.060       1696
3-6m                 0.339        433
6-12m                0.935        325
12-18m               1.000        204

  CLV by churn label:
                mean   median  count
churn_label                         
Active       7965.22  5780.18  14268
Churned      7715.21  5659.24    899
Dormant      8360.82  5907.95   1570


In [7]:
print("\n" + "="*70)
print("STEP 6 — SAVING FEATURE PLOTS")
print("="*70)
 
# Plot 1: Churn label distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = features["churn_label"].value_counts().sort_index()
bars   = ax.bar([label_map[k] for k in counts.index], counts.values,
                color=["steelblue", "coral", "grey"], edgecolor="white")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{val:,}\n({val/len(features)*100:.1f}%)",
            ha="center", va="bottom", fontsize=10)
ax.set_title("Churn Label Distribution", fontweight="bold")
ax.set_ylabel("Members")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_churn_label_distribution.png", dpi=120)
plt.close()
print(f"  Saved: plot_churn_label_distribution.png")
 
# Plot 2: Recency vs churn rate
fig, ax = plt.subplots(figsize=(8, 4))
cr = churn_by_recency["churn_rate"].reset_index()
ax.bar(cr["recency_bucket"].astype(str), cr["churn_rate"],
       color="coral", edgecolor="white")
ax.set_title("Churn Rate by Months Since Last Flight", fontweight="bold")
ax.set_xlabel("Months Since Last Flight (as of Jun 2018)")
ax.set_ylabel("Churn Rate")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_churn_rate_by_recency.png", dpi=120)
plt.close()
print(f"  Saved: plot_churn_rate_by_recency.png")
 
# Plot 3: CLV distribution by churn label (histogram, no scipy needed)
fig, ax = plt.subplots(figsize=(10, 4))
for label_code, color in [(0, "steelblue"), (1, "coral"), (2, "grey")]:
    subset = features[features["churn_label"] == label_code][CLV_COL]
    clipped = subset.clip(upper=subset.quantile(0.99))
    ax.hist(clipped, bins=40, alpha=0.4, color=color,
            label=label_map[label_code], density=True, edgecolor="none")
ax.set_title("CLV Distribution by Churn Label", fontweight="bold")
ax.set_xlabel("CLV")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_clv_by_churn_label.png", dpi=120)
plt.close()
print(f"  Saved: plot_clv_by_churn_label.png")
 
# Plot 4: Flight trend — bar chart of mean trend per group (cleaner than KDE)
fig, ax = plt.subplots(figsize=(7, 4))
trend_means = (features[features["churn_label"] < 2]
               .groupby("churn_label")["flight_trend"]
               .mean())
colors = ["steelblue", "coral"]
bars   = ax.bar([label_map[k] for k in trend_means.index],
                trend_means.values, color=colors, edgecolor="white")
for bar, val in zip(bars, trend_means.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.02 if val >= 0 else -0.08),
            f"{val:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Mean Flight Trend: Active vs Churned (positive = flying more recently)",
             fontweight="bold")
ax.set_ylabel("Mean Flight Trend (flights)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_flight_trend_churn.png", dpi=120)
plt.close()
print(f"  Saved: plot_flight_trend_churn.png")
 
 


STEP 6 — SAVING FEATURE PLOTS
  Saved: plot_churn_label_distribution.png
  Saved: plot_churn_rate_by_recency.png
  Saved: plot_clv_by_churn_label.png
  Saved: plot_flight_trend_churn.png


In [8]:
print("\n" + "="*70)
print("STEP 7 — SAVING MASTER FEATURE TABLE")
print("="*70)
 
# Drop helper columns not needed for modelling
drop_cols = ["recency_bucket", "flights_obs_window", "flights_churn_window"]
features.drop(columns=[c for c in drop_cols if c in features.columns], inplace=True)
 
out_path = f"{OUTPUT_DIR}/model_ready_features.csv"
features.to_csv(out_path, index=False)
 
print(f"\n  Saved: {out_path}")
print(f"  Shape: {features.shape[0]:,} members × {features.shape[1]} features")
 
# Print final feature list grouped by type
print(f"""
  FEATURE GROUPS:
  ────────────────────────────────────────────────────────
  Demographics  : clv, enrollment_tenure_months,
                  card_tier_encoded, salary_encoded,
                  education_encoded, gender_*, marital_status_*
 
  Recency       : months_since_last_flight
 
  Frequency     : flights_3m, flights_6m, flights_12m, flights_18m
                  active_months_3m, active_months_6m, ...
 
  Monetary      : pts_earned_18m, pts_redeemed_18m, distance_18m
 
  Trend         : flight_trend (recent 3m minus prior 3m)
 
  Engagement    : redemption_ratio, seasons_flown
 
  Flags         : never_flew_flag
 
  TARGET        : churn_label (0=Active, 1=Churned, 2=Dormant)
  ────────────────────────────────────────────────────────
""")
 
print("="*70)
print("PHASE 2 COMPLETE")
print("="*70)
print("""
  NEXT STEP → Phase 3: Churn Prediction Model
  Run: python phase3_churn_model.py
""")
 


STEP 7 — SAVING MASTER FEATURE TABLE

  Saved: Outputs/model_ready_features.csv
  Shape: 16,737 members × 35 features

  FEATURE GROUPS:
  ────────────────────────────────────────────────────────
  Demographics  : clv, enrollment_tenure_months,
                  card_tier_encoded, salary_encoded,
                  education_encoded, gender_*, marital_status_*
 
  Recency       : months_since_last_flight
 
  Frequency     : flights_3m, flights_6m, flights_12m, flights_18m
                  active_months_3m, active_months_6m, ...
 
  Monetary      : pts_earned_18m, pts_redeemed_18m, distance_18m
 
  Trend         : flight_trend (recent 3m minus prior 3m)
 
  Engagement    : redemption_ratio, seasons_flown
 
  Flags         : never_flew_flag
 
  TARGET        : churn_label (0=Active, 1=Churned, 2=Dormant)
  ────────────────────────────────────────────────────────

PHASE 2 COMPLETE

  NEXT STEP → Phase 3: Churn Prediction Model
  Run: python phase3_churn_model.py

